# 서울 랜드마크 이미지 분류


---
# ===== A 파트 (연주) =====
### 데이터 로드 / 경로 생성 / Dataset / DataLoader / 전처리

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch.nn as nn
import torch.optim as optim

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split


In [25]:
# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 중인 디바이스: {device}')

사용 중인 디바이스: cpu


## 1. 경로 설정 및 CSV 불러오기

> `train.csv`에는 파일명(`001.PNG`)과 라벨(0~9)이 있어.
> 실제 이미지를 불러오려면 파일명 앞에 폴더 경로를 붙여줘야 함

In [19]:
# 기본 경로 설정
BASE_DIR = '/content/drive/MyDrive/synaps_team_project'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
TEST_DIR  = os.path.join(BASE_DIR, 'test')

# CSV 불러오기
train_df = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(BASE_DIR, 'test.csv'))
sample_submission = pd.read_csv(os.path.join(BASE_DIR, 'sample_submission.csv'))

print('=== train.csv ===')
print(train_df.head())
print(f'\n총 학습 데이터: {len(train_df)}개')
print(f'클래스 종류: {sorted(train_df["label"].unique())}')
print(f'클래스 수: {train_df["label"].nunique()}개')
print('\n=== test.csv ===')
print(test_df.head())
print(f'\n총 테스트 데이터: {len(test_df)}개')

=== train.csv ===
  file_name  label
0   001.PNG      9
1   002.PNG      4
2   003.PNG      1
3   004.PNG      1
4   005.PNG      6

총 학습 데이터: 723개
클래스 종류: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
클래스 수: 10개

=== test.csv ===
  file_name
0   001.PNG
1   002.PNG
2   003.PNG
3   004.PNG
4   005.PNG

총 테스트 데이터: 199개


## 2. 이미지 파일 경로 생성

> 파일명에 폴더 경로를 붙여서 실제로 이미지를 열 수 있는 전체 경로를 만들어.

In [20]:
# 파일명 → 전체 경로로 변환
train_df['file_path'] = train_df['file_name'].apply(lambda x: os.path.join(TRAIN_DIR, x))
test_df['file_path']  = test_df['file_name'].apply(lambda x: os.path.join(TEST_DIR, x))

print('경로 생성 완료')
print(train_df[['file_name', 'file_path', 'label']].head())

경로 생성 완료
  file_name                                          file_path  label
0   001.PNG  /content/drive/MyDrive/synaps_team_project/tra...      9
1   002.PNG  /content/drive/MyDrive/synaps_team_project/tra...      4
2   003.PNG  /content/drive/MyDrive/synaps_team_project/tra...      1
3   004.PNG  /content/drive/MyDrive/synaps_team_project/tra...      1
4   005.PNG  /content/drive/MyDrive/synaps_team_project/tra...      6


## 3. 학습 / 검증 데이터 분리

> 학습: 검증데이터 = 8:2로 나눠서 검증 데이터로 중간 확인을 해!
> - train: 578개
> - validation: 145개

In [21]:
# 8:2 비율로 분리 (random_state=42 → 항상 같은 방식으로 나뉘게)
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']  # 클래스 비율 유지
)

# 인덱스 초기화
train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)

print(f'학습 데이터: {len(train_data)}개')
print(f'검증 데이터: {len(val_data)}개')

학습 데이터: 578개
검증 데이터: 145개


## 4. 이미지 전처리 (Transform)

> 이미지를 그대로 모델에 넣으면 안 되고 통일된 형태로 변환해야 해!
> - **Resize**: 모든 이미지를 224x224로 통일
> - **ToTensor**: 이미지를 숫자 배열(텐서)로 변환
> - **Normalize**: 픽셀값 범위를 줄여서 학습 안정화
>
> 학습용과 검증용 전처리를 다르게 하는 이유:
> → 학습할 때만 데이터 증강(뒤집기, 회전 등)을 적용해서 모델을 더 강하게 만들어!

In [22]:
# ImageNet 기준 평균/표준편차 (전이학습에서도 동일하게 사용)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# 학습용 전처리 (데이터 증강 포함)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),   # 좌우 뒤집기
    transforms.RandomRotation(10),       # ±10도 회전
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# 검증/테스트용 전처리 (증강 없이 그대로)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

print('전처리 설정 완료')

전처리 설정 완료


## 5. CustomDataset 구성

> PyTorch는 데이터를 불러올 때 정해진 형식이 있어.
> `__len__`: 데이터 개수 반환
> `__getitem__`: index를 주면 해당 이미지 + label 반환

In [23]:
class CustomDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        """
        df       : 파일 경로와 라벨이 담긴 DataFrame
        transform: 이미지 전처리
        is_test  : test 데이터면 True (라벨 없음)
        """
        self.df        = df
        self.transform = transform
        self.is_test   = is_test

    def __len__(self):
        # 데이터 총 개수 반환
        return len(self.df)

    def __getitem__(self, idx):
        # 이미지 경로에서 이미지 불러오기
        img_path = self.df.loc[idx, 'file_path']
        image    = Image.open(img_path).convert('RGB')  # RGB로 통일

        # 전처리 적용
        if self.transform:
            image = self.transform(image)

        # test 데이터는 라벨 없음
        if self.is_test:
            return image

        label = self.df.loc[idx, 'label']
        return image, label

print('CustomDataset 정의 완료')

CustomDataset 정의 완료


## 6. DataLoader 구성

> Dataset이 창고라면 DataLoader는 배달부야!
> 이미지를 batch_size 단위로 묶어서 모델에 전달해.
> - `batch_size=32`: 32개씩 묶어서 전달
> - `shuffle=True`: 학습할 때마다 순서 섞기 (과적합 방지)

In [24]:
BATCH_SIZE = 32

# Dataset 생성
train_dataset = CustomDataset(train_data, transform=train_transform)
val_dataset   = CustomDataset(val_data,   transform=val_transform)
test_dataset  = CustomDataset(test_df,    transform=val_transform, is_test=True)

# DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'train_loader: {len(train_loader)}개 배치')
print(f'val_loader:   {len(val_loader)}개 배치')
print(f'test_loader:  {len(test_loader)}개 배치')

# 정상 동작 확인
images, labels = next(iter(train_loader))
print(f'\n배치 이미지 shape: {images.shape}')  # (32, 3, 224, 224)
print(f'배치 라벨 shape:   {labels.shape}')   # (32,)

train_loader: 19개 배치
val_loader:   5개 배치
test_loader:  7개 배치

배치 이미지 shape: torch.Size([32, 3, 224, 224])
배치 라벨 shape:   torch.Size([32])


---
# ===== B 파트 (민진) =====
### CNN 모델 정의 / 학습 루프 / 검증 / test 예측 / submission 생성
